In [ ]:
import numpy as np
import os
import pandas as pd
import pickle
import torch

from scipy.spatial.distance import cosine
from tqdm import tqdm

In [ ]:
# Define the pickle file path
pickle_file_path = '../../BinDiffing-Measurement-Project/Results/data/raw_results/Hermessim/Dataset-6_hermessim/testing_Dataset-6.pkl'

# Load the pickle file
try:
    with open(pickle_file_path, 'rb') as file:
        data = pickle.load(file)

    # Print the type and some details of the data
    print(f"Data Type: {type(data)}")
    print("Data Preview:")
    print(data)
    
    if isinstance(data, torch.Tensor):
        print(f"Dimensions of the tensor: {data.shape}")

except Exception as e:
    print(f"Error loading the pickle file: {e}")

In [ ]:
def cosine_similarity(e1, e2):
    return 1 - cosine(e1, e2)

In [ ]:
def compute_cosine_similarity(df_input):
    sim_list = list()
    for idx, row in tqdm(df_input.iterrows()):

        if row['embeddings_1'] is np.nan or \
                row['embeddings_2'] is np.nan:
            print("[!] Missing value in (idx:{})".format(idx))
            sim_list.append(0)
            continue

        e1 = np.array([float(x) for x in row['embeddings_1'].split(";")])
        e2 = np.array([float(x) for x in row['embeddings_2'].split(";")])
        sim_list.append(cosine_similarity(e1, e2))
    return sim_list

In [ ]:
def compute_embedding_similarity(df_pairs, df_emb):
    
    df_emb = df_emb[['idb_path', 'fva', 'embeddings']]
    
    df_pairs = df_pairs.merge(df_emb,
                              how='left',
                              left_on=['idb_path_1', 'fva_1'],
                              right_on=['idb_path', 'fva'])
    df_pairs.rename(columns={'embeddings': 'embeddings_1'}, inplace=True)
    
    df_pairs = df_pairs.merge(df_emb,
                              how='left',
                              left_on=['idb_path_2', 'fva_2'],
                              right_on=['idb_path', 'fva'])
    df_pairs.rename(columns={'embeddings': 'embeddings_2'}, inplace=True)

    df_pairs['sim'] = compute_cosine_similarity(df_pairs)
    df_pairs = df_pairs[['idb_path_1','fva_1','idb_path_2','fva_2','sim']]
    return df_pairs

In [ ]:
def tensor_to_csv(tensor_path, csv_path, output_csv_path):
    # Load the tensor
    with open(tensor_path, 'rb') as f:
        embeddings = pickle.load(f)
    
    # Convert tensor to numpy array if needed
    if isinstance(embeddings, torch.Tensor):
        embeddings = embeddings.cpu().numpy()
    
    # Load the testing CSV
    df_meta = pd.read_csv(csv_path)
    
    # Verify row counts match
    if len(df_meta) != len(embeddings):
        raise ValueError(f"Row count mismatch: Metadata has {len(df_meta)} rows, embeddings have {len(embeddings)}")
    
    # Prepare output data
    output_data = []
    for idx in range(len(embeddings)):
        # Get metadata
        row_meta = df_meta.iloc[idx]
        idb_path = row_meta['idb_path']
        fva = row_meta['fva']
        
        # Get embedding values
        embedding_values = embeddings[idx]
        
        # Convert to semicolon-separated string with 8 decimal places
        embedding_str = ";".join(f"{val:.8f}" for val in embedding_values)
        
        output_data.append({
            'idb_path': idb_path,
            'fva': fva,
            'embeddings': embedding_str
        })
    
    # Create DataFrame and save
    df_output = pd.DataFrame(output_data)
    df_output.to_csv(output_csv_path, index=False)
    print(f"Saved {len(output_data)} rows to {output_csv_path}")

### Process Dataset-1 results

In [ ]:
DB1_PATH = "../../DBs/Dataset-1/pairs/testing/"
picke_path = "../data/raw_results/Hermessim/Dataset-1_hermessim/testing_Dataset-1.pkl"
embedding_path="../data/raw_results/Hermessim/Dataset-1_hermessim/embedding_Dataset-1_hermessim.csv"

if os.path.exists(embedding_path):
    print("CSV formated embedding file exists. No need to convert.")
else:
    print("CSV formated embedding file does not exist. Start converting.")
    tensor_to_csv(
            tensor_path=picke_path,
            csv_path="../../BinDiffing-Measurement-Project/DBs/Dataset-1/testing_Dataset-1.csv",
            output_csv_path=embedding_path
        )

df_emb = pd.read_csv(embedding_path)
df_pos = pd.read_csv(os.path.join(DB1_PATH, "pos_testing_Dataset-1.csv"), index_col=0)
df_neg = pd.read_csv(os.path.join(DB1_PATH, "neg_testing_Dataset-1.csv"), index_col=0)
df_pos_rank = pd.read_csv(os.path.join(DB1_PATH, "pos_rank_testing_Dataset-1.csv"), index_col=0)
df_neg_rank = pd.read_csv(os.path.join(DB1_PATH, "neg_rank_testing_Dataset-1.csv"), index_col=0)
df_pos = compute_embedding_similarity(df_pos, df_emb)
df_neg = compute_embedding_similarity(df_neg, df_emb)
df_pos_rank = compute_embedding_similarity(df_pos_rank, df_emb)
df_neg_rank = compute_embedding_similarity(df_neg_rank, df_emb)
df_pos.to_csv("../data/Dataset-1/pos_testing_{}.csv".format('Dataset-1_hermessim'), index=False)
df_neg.to_csv("../data/Dataset-1/neg_testing_{}.csv".format('Dataset-1_hermessim'), index=False)
df_pos_rank.to_csv("../data/Dataset-1/pos_rank_testing_{}.csv".format('Dataset-1_hermessim'), index=False)
df_neg_rank.to_csv("../data/Dataset-1/neg_rank_testing_{}.csv".format('Dataset-1_hermessim'), index=False)

### Process Dataset-2 results

In [ ]:
DB2_PATH = "../../DBs/Dataset-2/pairs/testing/"
picke_path = "../data/raw_results/Hermessim/Dataset-2_hermessim/testing_Dataset-2.pkl"
embedding_path="../data/raw_results/Hermessim/Dataset-2_hermessim/embedding_Dataset-2_hermessim.csv"

if os.path.exists(embedding_path):
    print("CSV formated embedding file exists. No need to convert.")
else:
    print("CSV formated embedding file does not exist. Start converting.")
    tensor_to_csv(
            tensor_path=picke_path,
            csv_path="../../BinDiffing-Measurement-Project/DBs/Dataset-2/testing_Dataset-2.csv",
            output_csv_path=embedding_path
        )

df_emb = pd.read_csv(embedding_path)
df_pos = pd.read_csv(os.path.join(DB2_PATH, "pos_testing_Dataset-2.csv"), index_col=0)
df_neg = pd.read_csv(os.path.join(DB2_PATH, "neg_testing_Dataset-2.csv"), index_col=0)
df_pos_rank = pd.read_csv(os.path.join(DB2_PATH, "pos_rank_testing_Dataset-2.csv"), index_col=0)
df_neg_rank = pd.read_csv(os.path.join(DB2_PATH, "neg_rank_testing_Dataset-2.csv"), index_col=0)
df_pos = compute_embedding_similarity(df_pos, df_emb)
df_neg = compute_embedding_similarity(df_neg, df_emb)
df_pos_rank = compute_embedding_similarity(df_pos_rank, df_emb)
df_neg_rank = compute_embedding_similarity(df_neg_rank, df_emb)
df_pos.to_csv("../data/Dataset-2/pos_testing_{}.csv".format('Dataset-2_hermessim'), index=False)
df_neg.to_csv("../data/Dataset-2/neg_testing_{}.csv".format('Dataset-2_hermessim'), index=False)
df_pos_rank.to_csv("../data/Dataset-2/pos_rank_testing_{}.csv".format('Dataset-2_hermessim'), index=False)
df_neg_rank.to_csv("../data/Dataset-2/neg_rank_testing_{}.csv".format('Dataset-2_hermessim'), index=False)

### Process Dataset-3 results

In [ ]:
DB3_PATH = "../../DBs/Dataset-3/pairs/testing/"
picke_path = "../data/raw_results/Hermessim/Dataset-3_hermessim/testing_Dataset-3.pkl"
embedding_path="../data/raw_results/Hermessim/Dataset-3_hermessim/embedding_Dataset-3_hermessim.csv"

if os.path.exists(embedding_path):
    print("CSV formated embedding file exists. No need to convert.")
else:
    print("CSV formated embedding file does not exist. Start converting.")
    tensor_to_csv(
            tensor_path=picke_path,
            csv_path="../../BinDiffing-Measurement-Project/DBs/Dataset-3/testing_Dataset-3.csv",
            output_csv_path=embedding_path
        )

df_emb = pd.read_csv(embedding_path)
df_pos = pd.read_csv(os.path.join(DB3_PATH, "pos_testing_Dataset-3.csv"), index_col=0)
df_neg = pd.read_csv(os.path.join(DB3_PATH, "neg_testing_Dataset-3.csv"), index_col=0)
df_pos_rank = pd.read_csv(os.path.join(DB3_PATH, "pos_rank_testing_Dataset-3.csv"), index_col=0)
df_neg_rank = pd.read_csv(os.path.join(DB3_PATH, "neg_rank_testing_Dataset-3.csv"), index_col=0)
df_pos = compute_embedding_similarity(df_pos, df_emb)
df_neg = compute_embedding_similarity(df_neg, df_emb)
df_pos_rank = compute_embedding_similarity(df_pos_rank, df_emb)
df_neg_rank = compute_embedding_similarity(df_neg_rank, df_emb)
df_pos.to_csv("../data/Dataset-3/pos_testing_{}.csv".format('Dataset-3_hermessim'), index=False)
df_neg.to_csv("../data/Dataset-3/neg_testing_{}.csv".format('Dataset-3_hermessim'), index=False)
df_pos_rank.to_csv("../data/Dataset-3/pos_rank_testing_{}.csv".format('Dataset-3_hermessim'), index=False)
df_neg_rank.to_csv("../data/Dataset-3/neg_rank_testing_{}.csv".format('Dataset-3_hermessim'), index=False)

### Process Dataset-4 results

In [ ]:
DB4_PATH = "../../DBs/Dataset-4/pairs/testing/"
picke_path = "../data/raw_results/Hermessim/Dataset-4_hermessim/testing_Dataset-4.pkl"
embedding_path="../data/raw_results/Hermessim/Dataset-4_hermessim/embedding_Dataset-4_hermessim.csv"

if os.path.exists(embedding_path):
    print("CSV formated embedding file exists. No need to convert.")
else:
    print("CSV formated embedding file does not exist. Start converting.")
    tensor_to_csv(
            tensor_path=picke_path,
            csv_path="../../BinDiffing-Measurement-Project/DBs/Dataset-4/testing_Dataset-4.csv",
            output_csv_path=embedding_path
        )

df_emb = pd.read_csv(embedding_path)
df_pos = pd.read_csv(os.path.join(DB4_PATH, "pos_testing_Dataset-4.csv"), index_col=0)
df_neg = pd.read_csv(os.path.join(DB4_PATH, "neg_testing_Dataset-4.csv"), index_col=0)
df_pos_rank = pd.read_csv(os.path.join(DB4_PATH, "pos_rank_testing_Dataset-4.csv"), index_col=0)
df_neg_rank = pd.read_csv(os.path.join(DB4_PATH, "neg_rank_testing_Dataset-4.csv"), index_col=0)
df_pos = compute_embedding_similarity(df_pos, df_emb)
df_neg = compute_embedding_similarity(df_neg, df_emb)
df_pos_rank = compute_embedding_similarity(df_pos_rank, df_emb)
df_neg_rank = compute_embedding_similarity(df_neg_rank, df_emb)
df_pos.to_csv("../data/Dataset-4/pos_testing_{}.csv".format('Dataset-4_hermessim'), index=False)
df_neg.to_csv("../data/Dataset-4/neg_testing_{}.csv".format('Dataset-4_hermessim'), index=False)
df_pos_rank.to_csv("../data/Dataset-4/pos_rank_testing_{}.csv".format('Dataset-4_hermessim'), index=False)
df_neg_rank.to_csv("../data/Dataset-4/neg_rank_testing_{}.csv".format('Dataset-4_hermessim'), index=False)

### Process Dataset-5 results

In [ ]:
DB5_PATH = "../../DBs/Dataset-5/pairs/testing/"
picke_path = "../data/raw_results/Hermessim/Dataset-5_hermessim/testing_Dataset-5.pkl"
embedding_path="../data/raw_results/Hermessim/Dataset-5_hermessim/embedding_Dataset-5_hermessim.csv"

if os.path.exists(embedding_path):
    print("CSV formated embedding file exists. No need to convert.")
else:
    print("CSV formated embedding file does not exist. Start converting.")
    tensor_to_csv(
            tensor_path=picke_path,
            csv_path="../../BinDiffing-Measurement-Project/DBs/Dataset-5/testing_Dataset-5.csv",
            output_csv_path=embedding_path
        )

df_emb = pd.read_csv(embedding_path)
df_pos = pd.read_csv(os.path.join(DB5_PATH, "pos_testing_Dataset-5.csv"), index_col=0)
df_neg = pd.read_csv(os.path.join(DB5_PATH, "neg_testing_Dataset-5.csv"), index_col=0)
df_pos_rank = pd.read_csv(os.path.join(DB5_PATH, "pos_rank_testing_Dataset-5.csv"), index_col=0)
df_neg_rank = pd.read_csv(os.path.join(DB5_PATH, "neg_rank_testing_Dataset-5.csv"), index_col=0)
df_pos = compute_embedding_similarity(df_pos, df_emb)
df_neg = compute_embedding_similarity(df_neg, df_emb)
df_pos_rank = compute_embedding_similarity(df_pos_rank, df_emb)
df_neg_rank = compute_embedding_similarity(df_neg_rank, df_emb)
df_pos.to_csv("../data/Dataset-5/pos_testing_{}.csv".format('Dataset-5_hermessim'), index=False)
df_neg.to_csv("../data/Dataset-5/neg_testing_{}.csv".format('Dataset-5_hermessim'), index=False)
df_pos_rank.to_csv("../data/Dataset-5/pos_rank_testing_{}.csv".format('Dataset-5_hermessim'), index=False)
df_neg_rank.to_csv("../data/Dataset-5/neg_rank_testing_{}.csv".format('Dataset-5_hermessim'), index=False)

### Process Dataset-6 results

In [ ]:
DB6_PATH = "../../DBs/Dataset-6/pairs/testing/"
picke_path = "../data/raw_results/Hermessim/Dataset-6_hermessim/testing_Dataset-6.pkl"
embedding_path="../data/raw_results/Hermessim/Dataset-6_hermessim/embedding_Dataset-6_hermessim.csv"

if os.path.exists(embedding_path):
    print("CSV formated embedding file exists. No need to convert.")
else:
    print("CSV formated embedding file does not exist. Start converting.")
    tensor_to_csv(
            tensor_path=picke_path,
            csv_path="../../BinDiffing-Measurement-Project/DBs/Dataset-6/testing_Dataset-6.csv",
            output_csv_path=embedding_path
        )

df_emb = pd.read_csv(embedding_path)
df_pos = pd.read_csv(os.path.join(DB6_PATH, "pos_testing_Dataset-6.csv"), index_col=0)
df_neg = pd.read_csv(os.path.join(DB6_PATH, "neg_testing_Dataset-6.csv"), index_col=0)
df_pos_rank = pd.read_csv(os.path.join(DB6_PATH, "pos_rank_testing_Dataset-6.csv"), index_col=0)
df_neg_rank = pd.read_csv(os.path.join(DB6_PATH, "neg_rank_testing_Dataset-6.csv"), index_col=0)
df_pos = compute_embedding_similarity(df_pos, df_emb)
df_neg = compute_embedding_similarity(df_neg, df_emb)
df_pos_rank = compute_embedding_similarity(df_pos_rank, df_emb)
df_neg_rank = compute_embedding_similarity(df_neg_rank, df_emb)
df_pos.to_csv("../data/Dataset-6/pos_testing_{}.csv".format('Dataset-6_hermessim'), index=False)
df_neg.to_csv("../data/Dataset-6/neg_testing_{}.csv".format('Dataset-6_hermessim'), index=False)
df_pos_rank.to_csv("../data/Dataset-6/pos_rank_testing_{}.csv".format('Dataset-6_hermessim'), index=False)
df_neg_rank.to_csv("../data/Dataset-6/neg_rank_testing_{}.csv".format('Dataset-6_hermessim'), index=False)

### Process Dataset-7 results

In [ ]:
DB7_PATH = "../../DBs/Dataset-7/pairs/testing/"
picke_path = "../data/raw_results/Hermessim/Dataset-7_hermessim/testing_Dataset-7.pkl"
embedding_path="../data/raw_results/Hermessim/Dataset-7_hermessim/embedding_Dataset-7_hermessim.csv"

if os.path.exists(embedding_path):
    print("CSV formated embedding file exists. No need to convert.")
else:
    print("CSV formated embedding file does not exist. Start converting.")
    tensor_to_csv(
            tensor_path=picke_path,
            csv_path="../../BinDiffing-Measurement-Project/DBs/Dataset-7/testing_Dataset-7.csv",
            output_csv_path=embedding_path
        )

df_emb = pd.read_csv(embedding_path)
df_pos = pd.read_csv(os.path.join(DB7_PATH, "pos_testing_Dataset-7.csv"), index_col=0)
df_neg = pd.read_csv(os.path.join(DB7_PATH, "neg_testing_Dataset-7.csv"), index_col=0)
df_pos_rank = pd.read_csv(os.path.join(DB7_PATH, "pos_rank_testing_Dataset-7.csv"), index_col=0)
df_neg_rank = pd.read_csv(os.path.join(DB7_PATH, "neg_rank_testing_Dataset-7.csv"), index_col=0)
df_pos = compute_embedding_similarity(df_pos, df_emb)
df_neg = compute_embedding_similarity(df_neg, df_emb)
df_pos_rank = compute_embedding_similarity(df_pos_rank, df_emb)
df_neg_rank = compute_embedding_similarity(df_neg_rank, df_emb)
df_pos.to_csv("../data/Dataset-7/pos_testing_{}.csv".format('Dataset-7_hermessim'), index=False)
df_neg.to_csv("../data/Dataset-7/neg_testing_{}.csv".format('Dataset-7_hermessim'), index=False)
df_pos_rank.to_csv("../data/Dataset-7/pos_rank_testing_{}.csv".format('Dataset-7_hermessim'), index=False)
df_neg_rank.to_csv("../data/Dataset-7/neg_rank_testing_{}.csv".format('Dataset-7_hermessim'), index=False)